Calculate each user's average session time, where a session is defined as the time difference between a page_load and a page_exit. Assume each user has only one session per day. If there are multiple page_load or page_exit events on the same day, use only the latest page_load and the earliest page_exit. Only consider sessions where the page_load occurs before the page_exit on the same day. Output the user_id and their average session time.

Approach

For each user_id + date:
Get latest page_load
Get earliest page_exit
Keep only valid sessions where:
page_load < page_exit
Compute session duration
Take average per user

In [0]:
%sql
WITH session_events AS (
    SELECT 
        user_id,
        DATE(event_time) AS event_date,
        MAX(CASE WHEN event_type = 'page_load' THEN event_time END) AS last_load,
        MIN(CASE WHEN event_type = 'page_exit' THEN event_time END) AS first_exit
    FROM events
    GROUP BY user_id, DATE(event_time)
),
valid_sessions AS (
    SELECT 
        user_id,
        TIMESTAMPDIFF(SECOND, last_load, first_exit) AS session_time
    FROM session_events
    WHERE last_load IS NOT NULL
      AND first_exit IS NOT NULL
      AND last_load < first_exit
)
SELECT 
    user_id,
    AVG(session_time) AS avg_session_time_seconds
FROM valid_sessions
GROUP BY user_id;

In [0]:
from pyspark.sql import functions as F

# Step 1: Extract date
df = df.withColumn("event_date", F.to_date("event_time"))

# Step 2: Aggregate per user per day
session_df = df.groupBy("user_id", "event_date").agg(
    F.max(F.when(F.col("event_type") == "page_load", F.col("event_time"))).alias("last_load"),
    F.min(F.when(F.col("event_type") == "page_exit", F.col("event_time"))).alias("first_exit")
)

# Step 3: Filter valid sessions
valid_sessions = session_df.filter(
    (F.col("last_load").isNotNull()) &
    (F.col("first_exit").isNotNull()) &
    (F.col("last_load") < F.col("first_exit"))
)

# Step 4: Calculate session duration
valid_sessions = valid_sessions.withColumn(
    "session_time",
    F.unix_timestamp("first_exit") - F.unix_timestamp("last_load")
)

# Step 5: Average per user
result = valid_sessions.groupBy("user_id").agg(
    F.avg("session_time").alias("avg_session_time_seconds")
)

result.show()